# 04 · Gold

Silver → business tables in `data/gold/`, fully recomputed on each run:

| Table | Grain |
|---|---|
| `enem_fact_candidates` | one row per candidate, with average score and performance tier |
| `enem_performance_by_state` | year × state |
| `enem_performance_by_school` | year × state × school type × gender |

**Business rule:** `average_score` only exists for candidates who attended all four objective exams.
Absent candidates get the `Absent` tier and stay out of the averages instead of counting as zero.

In [1]:
%run ./00_setup.ipynb

:: loading settings :: url = jar:file:/home/gabriel/Documents/GitHub/ETL-com-PySpark/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/gabriel/Documents/GitHub/ETL-com-PySpark/.ivy2/cache
The jars for the packages stored in: /home/gabriel/Documents/GitHub/ETL-com-PySpark/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ff5d9b31-095f-4c85-88bc-f769a81f250f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 91ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------------------------

Spark 3.5.3 ready


In [2]:
from pyspark.sql.functions import avg, col, count, round as spark_round, stddev, when

EXAM_SCORES = ["score_natural_sciences", "score_humanities", "score_languages", "score_math", "score_essay"]

silver = spark.read.format("delta").load(SILVER_TABLE)

### Fact table: candidates

In [3]:
present_all_exams = (
    (col("presence_natural_sciences") == 1)
    & (col("presence_humanities") == 1)
    & (col("presence_languages") == 1)
    & (col("presence_math") == 1)
)
average = sum(col(c) for c in EXAM_SCORES) / len(EXAM_SCORES)

fact_candidates = (
    silver.withColumn("is_present_all_exams", present_all_exams)
    .withColumn("average_score", when(col("is_present_all_exams"), spark_round(average, 2)))
    .withColumn(
        "performance_tier",
        when(col("average_score").isNull(), "Absent")
        .when(col("average_score") >= 750, "Excellent")
        .when(col("average_score") >= 600, "Good")
        .when(col("average_score") >= 450, "Average")
        .otherwise("Below Average"),
    )
)

### Aggregates: by state and by school type

In [4]:
# avg() ignores nulls, so absent candidates don't pull the averages down
score_averages = [spark_round(avg(c), 2).alias(c.replace("score_", "avg_")) for c in EXAM_SCORES]

performance_by_state = fact_candidates.groupBy("year", "exam_state").agg(
    count("*").alias("total_enrolled"),
    count(when(col("is_present_all_exams"), True)).alias("total_present_all_exams"),
    *score_averages,
    spark_round(stddev("score_essay"), 2).alias("stddev_essay"),
    spark_round(avg("average_score"), 2).alias("overall_avg_score"),
)

school_type = (
    when(col("school_type") == 1, "Not Specified")
    .when(col("school_type") == 2, "Public")
    .when(col("school_type") == 3, "Private")
    .otherwise("Other")
)

performance_by_school = (
    fact_candidates.withColumn("school_type_description", school_type)
    .groupBy("year", "exam_state", "school_type_description", "gender")
    .agg(
        count("*").alias("total_candidates"),
        *score_averages,
        spark_round(avg("average_score"), 2).alias("overall_avg_score"),
    )
)

Data quality check: absent candidates must not have an average score.

In [5]:
assert fact_candidates.filter(~col("is_present_all_exams") & col("average_score").isNotNull()).isEmpty()
print("Quality check passed")

26/11/25 14:23:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Quality check passed


In [6]:
gold_tables = {
    # name: (DataFrame, partition columns) - only the candidate-level fact table is large enough to partition
    "enem_fact_candidates": (fact_candidates, ["year", "exam_state"]),
    "enem_performance_by_state": (performance_by_state, []),
    "enem_performance_by_school": (performance_by_school, []),
}

for name, (df, partitions) in gold_tables.items():
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  # Gold is recomputed: allow schema/partition changes
        .partitionBy(*partitions)
        .save(str(GOLD_DIR / name))
    )
    print(f"Saved {name}")

Saved enem_fact_candidates
Saved enem_performance_by_state
Saved enem_performance_by_school


### Results

In [7]:
gold = {name: spark.read.format("delta").load(str(GOLD_DIR / name)) for name in gold_tables}

gold["enem_performance_by_state"].orderBy(col("overall_avg_score").desc()).show(truncate=False)

+----+----------+--------------+-----------------------+--------------------+--------------+-------------+--------+---------+------------+-----------------+
|year|exam_state|total_enrolled|total_present_all_exams|avg_natural_sciences|avg_humanities|avg_languages|avg_math|avg_essay|stddev_essay|overall_avg_score|
+----+----------+--------------+-----------------------+--------------------+--------------+-------------+--------+---------+------------+-----------------+
|2020|PR        |1             |1                      |720.6               |750.3         |690.4        |780.9   |920.0    |NULL        |772.44           |
|2020|SP        |3             |3                      |630.47              |667.1         |620.3        |693.77  |806.67   |133.17      |683.66           |
|2020|CE        |2             |2                      |635.2               |655.2         |615.6        |700.6   |770.0    |14.14       |675.32           |
|2020|RJ        |2             |2                      |59

In [8]:
gold["enem_performance_by_school"].show(10, truncate=False)

+----+----------+-----------------------+------+----------------+--------------------+--------------+-------------+--------+---------+-----------------+
|year|exam_state|school_type_description|gender|total_candidates|avg_natural_sciences|avg_humanities|avg_languages|avg_math|avg_essay|overall_avg_score|
+----+----------+-----------------------+------+----------------+--------------------+--------------+-------------+--------+---------+-----------------+
|2020|SP        |Public                 |F     |1               |560.5               |620.0         |580.4        |640.2   |720.0    |624.22           |
|2020|SP        |Private                |F     |1               |740.8               |770.5         |710.3        |810.2   |960.0    |798.36           |
|2020|SP        |Public                 |M     |1               |590.1               |610.8         |570.2        |630.9   |740.0    |628.4            |
|2020|CE        |Public                 |M     |1               |630.1            

In [9]:
gold["enem_fact_candidates"].select(
    "registration_id", "exam_state", "is_present_all_exams", "average_score", "performance_tier"
).show(10)

+---------------+----------+--------------------+-------------+----------------+
|registration_id|exam_state|is_present_all_exams|average_score|performance_tier|
+---------------+----------+--------------------+-------------+----------------+
|   200001000001|        SP|                true|       624.22|            Good|
|   200001000011|        SP|                true|        628.4|            Good|
|   200001000016|        SP|                true|       798.36|       Excellent|
|   200001000009|        CE|                true|       682.34|            Good|
|   200001000018|        CE|                true|        668.3|            Good|
|   200001000005|        RS|                true|       652.34|            Good|
|   200001000015|        RS|                true|        614.3|            Good|
|   200001000008|        PE|                true|       486.38|         Average|
|   200001000017|        PE|               false|         NULL|          Absent|
|   200001000010|        GO|

Any table can be queried with SQL through a temporary view.

In [10]:
gold["enem_fact_candidates"].createOrReplaceTempView("fact_candidates")

spark.sql("""
    SELECT performance_tier, COUNT(*) AS candidates, ROUND(AVG(average_score), 2) AS avg_score
    FROM fact_candidates
    GROUP BY performance_tier
    ORDER BY avg_score DESC NULLS LAST
""").show()

+----------------+----------+---------+
|performance_tier|candidates|avg_score|
+----------------+----------+---------+
|       Excellent|         2|    785.4|
|            Good|         9|   663.45|
|         Average|         7|   526.94|
|          Absent|         2|     NULL|
+----------------+----------+---------+



In [11]:
spark.stop()